# 01 — Data Verification
Checks that all datasets and embeddings are in the right place with the right structure.
**Does not download anything.** Just reads and reports.

In [50]:
import sys, json, pickle
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path('../../').resolve()))
from src.utils.paths import *

results = []

def check(label, condition, detail=''):
    sym = '  ✓' if condition else '  ✗'
    msg = f"{sym}  {label}"
    if detail: msg += f"   ({detail})"
    print(msg)
    results.append(condition)
    return condition

print(f'  ROOT : {ROOT}')
print(f'  DATA : {DATA}')
print(f'  CACHE: {CACHE}')

  ROOT : F:\GitHub\eve-rag
  DATA : F:\GitHub\eve-rag\data
  CACHE: F:\GitHub\eve-rag\cache


## 1. NewsCLIPpings — pair JSONs

In [51]:
print('NewsCLIPpings pair JSONs')
print(f'  Location: {NEWSCLIP}/')
print()

for fname, min_count in [('train.json', 57000), ('val.json', 7000), ('test.json', 7000)]:
    fpath = NEWSCLIP / fname
    if not check(f'{fname} exists', fpath.exists(), f'{fpath}'):
        continue
    try:
        raw = json.loads(fpath.read_text())
        pairs = raw.get('annotations', raw) if isinstance(raw, dict) else raw
        n = len(pairs)
        check(f'{fname} count', n >= min_count, f'{n:,} pairs  (expected >= {min_count:,})')
        s = pairs[0]
        # NewsCLIPpings uses 'falsified' as label; 'caption' lives in VisualNews, not here
        label_key = 'falsified' if 'falsified' in s else 'label'
        check(f'{fname} schema',
              all(k in s for k in ['id', 'image_id', label_key]),
              f'keys: {list(s.keys())}')
        ooc = sum(1 for p in pairs if p.get(label_key, 0) == 1)
        print(f'       OOC: {ooc:,}  Genuine: {n-ooc:,}  ({ooc/n*100:.1f}% OOC)')
    except Exception as e:
        check(f'{fname} readable', False, str(e))
    print()

NewsCLIPpings pair JSONs
  Location: F:\GitHub\eve-rag\data\newsclippings/

  ✓  train.json exists   (F:\GitHub\eve-rag\data\newsclippings\train.json)
  ✓  train.json count   (71,072 pairs  (expected >= 57,000))
  ✓  train.json schema   (keys: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified'])
       OOC: 35,536  Genuine: 35,536  (50.0% OOC)

  ✓  val.json exists   (F:\GitHub\eve-rag\data\newsclippings\val.json)
  ✓  val.json count   (7,024 pairs  (expected >= 7,000))
  ✓  val.json schema   (keys: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified'])
       OOC: 3,512  Genuine: 3,512  (50.0% OOC)

  ✓  test.json exists   (F:\GitHub\eve-rag\data\newsclippings\test.json)
  ✓  test.json count   (7,264 pairs  (expected >= 7,000))
  ✓  test.json schema   (keys: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified'])
       OOC: 3,632  Genuine: 3,632  (50.0% OOC)



## 2. NewsCLIPpings — CLIP & SBERT embeddings

In [52]:
print('NewsCLIPpings embeddings')
print(f'  Location: {CLIP_EMB}/')
print(f'  Source  : https://huggingface.co/g-luo/news-clippings/tree/main/embeddings')
print()

emb_types = {
    'clip_image_embeddings': 'CLIP image  (512-dim)',
    'clip_text_embeddings' : 'CLIP text   (512-dim)',
    'sbert_embeddings'     : 'SBERT text  (768-dim)',
}

for folder, desc in emb_types.items():
    print(f'  {desc}')
    for split in ['train', 'val', 'test']:
        fpath = CLIP_EMB / folder / f'{split}.pkl'
        if not check(f'    {folder}/{split}.pkl', fpath.exists()):
            continue
        try:
            mb = fpath.stat().st_size / 1e6
            data = pickle.load(open(fpath, 'rb'))
            n = len(data)
            sample_val = next(iter(data.values()))
            dim = sample_val.shape[0] if hasattr(sample_val, 'shape') else len(sample_val)
            check(f'    {folder}/{split}.pkl valid',
                  n > 1000,
                  f'{n:,} entries  dim={dim}  {mb:.0f} MB')
        except Exception as e:
            check(f'    {folder}/{split}.pkl readable', False, str(e))
    print()

NewsCLIPpings embeddings
  Location: F:\GitHub\eve-rag\cache\clip_embeddings/
  Source  : https://huggingface.co/g-luo/news-clippings/tree/main/embeddings

  CLIP image  (512-dim)
  ✓      clip_image_embeddings/train.pkl
  ✓      clip_image_embeddings/train.pkl valid   (473,663 entries  dim=512  1001 MB)
  ✓      clip_image_embeddings/val.pkl
  ✓      clip_image_embeddings/val.pkl valid   (50,000 entries  dim=512  106 MB)
  ✓      clip_image_embeddings/test.pkl
  ✓      clip_image_embeddings/test.pkl valid   (50,000 entries  dim=512  106 MB)

  CLIP text   (512-dim)
  ✓      clip_text_embeddings/train.pkl
  ✓      clip_text_embeddings/train.pkl valid   (385,850 entries  dim=512  816 MB)
  ✓      clip_text_embeddings/val.pkl
  ✓      clip_text_embeddings/val.pkl valid   (40,864 entries  dim=512  86 MB)
  ✓      clip_text_embeddings/test.pkl
  ✓      clip_text_embeddings/test.pkl valid   (40,915 entries  dim=512  86 MB)

  SBERT text  (768-dim)
  ✓      sbert_embeddings/train.pkl
  ✓    

## 3. VERITE

In [53]:
print('VERITE')
print(f'  Location: {VERITE}/')
print()

import csv
from collections import Counter

verite_csv  = VERITE / 'VERITE.csv'
verite_art  = VERITE / 'VERITE_articles.csv'
verite_json = VERITE / 'pairs.json'

if verite_csv.exists():
    verite_file = verite_csv
elif verite_json.exists():
    verite_file = verite_json
elif verite_art.exists():
    verite_file = verite_art
else:
    verite_file = None

if check('annotations exist', verite_file is not None,
         f'looked for VERITE.csv / pairs.json in {VERITE}'):
    try:
        if verite_file.suffix == '.csv':
            rows = list(csv.DictReader(open(verite_file, encoding='utf-8')))
            check('row count', len(rows) >= 900,
                  f'{len(rows)} rows  (expected ~1000)')
            print(f'     Columns : {list(rows[0].keys())}')
            label_col = next((k for k in rows[0] if 'label' in k.lower()), None)
            if label_col:
                print(f'     Labels  : {dict(Counter(r[label_col] for r in rows))}')
        else:
            pairs = json.loads(verite_file.read_text())
            check('row count', len(pairs) >= 900,
                  f'{len(pairs)} pairs  (expected ~1000)')
    except Exception as e:
        check('annotations readable', False, str(e))

# CLIP embeddings (ViT-L/14 — better than NewsCLIPpings ViT-B/32)
npy_img  = CLIP_EMB / 'verite' / 'image_embeddings.npy'
npy_text = CLIP_EMB / 'verite' / 'text_embeddings.npy'
check('CLIP image embeddings (.npy)', npy_img.exists(),  str(npy_img))
check('CLIP text  embeddings (.npy)', npy_text.exists(), str(npy_text))
if npy_img.exists():
    import numpy as np
    arr = np.load(str(npy_img))
    print(f'     image emb shape: {arr.shape}  (expected ~[1000, 768])')
if npy_text.exists():
    import numpy as np
    arr = np.load(str(npy_text))
    print(f'     text  emb shape: {arr.shape}')

# Images (optional for now — only needed for MLLM notebooks)
img_dir = VERITE / 'images'
imgs = list(img_dir.rglob('*.jpg')) + list(img_dir.rglob('*.png')) + list(img_dir.rglob('*.jpeg')) if img_dir.exists() else []
check('images folder exists', img_dir.exists(), str(img_dir))
print(f'     Images found : {len(imgs)}  (optional — needed only for MLLM stage)')


VERITE
  Location: F:\GitHub\eve-rag\data\verite/

  ✓  annotations exist   (looked for VERITE.csv / pairs.json in F:\GitHub\eve-rag\data\verite)
  ✓  row count   (1001 rows  (expected ~1000))
     Columns : ['', 'caption', 'image_path', 'label']
     Labels  : {'true': 338, 'miscaptioned': 338, 'out-of-context': 325}
  ✓  CLIP image embeddings (.npy)   (F:\GitHub\eve-rag\cache\clip_embeddings\verite\image_embeddings.npy)
  ✓  CLIP text  embeddings (.npy)   (F:\GitHub\eve-rag\cache\clip_embeddings\verite\text_embeddings.npy)
     image emb shape: (1001, 768)  (expected ~[1000, 768])
     text  emb shape: (1001, 768)
  ✓  images folder exists   (F:\GitHub\eve-rag\data\verite\images)
     Images found : 27  (optional — needed only for MLLM stage)


## 4. MMFakeBench

In [54]:
MMFAKE = DATA / 'mmfakebench'

print('MMFakeBench')
print(f'  Location: {MMFAKE}/')
print(f'  Source  : https://huggingface.co/datasets/Xuannan/MMFakeBench  (fill usage form)')
print()

for fname, min_n in [('MMFakeBench_val.json', 500), ('MMFakeBench_test.json', 5000)]:
    fpath = MMFAKE / fname
    if not check(f'{fname} exists', fpath.exists()):
        continue
    try:
        raw = json.loads(fpath.read_text())
        pairs = raw if isinstance(raw, list) else list(raw.values())
        n = len(pairs)
        check(f'{fname} count', n >= min_n, f'{n:,} samples  (expected >= {min_n:,})')
        s = pairs[0]
        check(f'{fname} schema',
              all(k in s for k in ['text', 'image_path', 'gt_answers']),
              f'keys: {list(s.keys())}')
        print(f'     Labels  : {dict(Counter(p["gt_answers"] for p in pairs))}')
        ooc = [p for p in pairs if 'cross_modal' in p.get('fake_cls', '').lower()
               or 'ooc' in p.get('fake_cls', '').lower()]
        print(f'     OOC-type: {len(ooc):,}  (cross_modal_consistency category)')
    except Exception as e:
        check(f'{fname} readable', False, str(e))
    print()

# images can be in images/, fake/, real/, or just alongside JSON
total_imgs = sum(
    len(list(d.rglob('*.jpg'))) + len(list(d.rglob('*.png')))
    for d in [MMFAKE, MMFAKE/'images', MMFAKE/'fake', MMFAKE/'real']
    if d.exists()
)
check('images present', total_imgs >= 1000, f'{total_imgs:,} images found')

MMFakeBench
  Location: F:\GitHub\eve-rag\data\mmfakebench/
  Source  : https://huggingface.co/datasets/Xuannan/MMFakeBench  (fill usage form)

  ✓  MMFakeBench_val.json exists
  ✓  MMFakeBench_val.json count   (1,000 samples  (expected >= 500))
  ✓  MMFakeBench_val.json schema   (keys: ['text', 'image_path', 'text_source', 'image_source', 'gt_answers', 'fake_cls'])
     Labels  : {'True': 300, 'Fake': 700}
     OOC-type: 0  (cross_modal_consistency category)

  ✓  MMFakeBench_test.json exists
  ✓  MMFakeBench_test.json count   (10,000 samples  (expected >= 5,000))
  ✓  MMFakeBench_test.json schema   (keys: ['text', 'image_path', 'text_source', 'image_source', 'gt_answers', 'fake_cls'])
     Labels  : {'True': 3000, 'Fake': 7000}
     OOC-type: 0  (cross_modal_consistency category)

  ✓  images present   (11,000 images found)


True

## Summary

In [55]:
passed = sum(results)
total  = len(results)

print('=' * 50)
print(f'  {passed} / {total} checks passed')
if passed == total:
    print('  ✓  All data verified — ready for EDA.')
    print('  Next: notebooks/01_setup/eda.ipynb')
else:
    print(f'  ✗  {total - passed} check(s) failed — see above.')
    print()
    print('  Download links:')
    print('  NewsCLIPpings JSONs  → huggingface.co/g-luo/news-clippings/tree/main/data/merged_balanced')
    print('  CLIP embeddings      → huggingface.co/g-luo/news-clippings/tree/main/embeddings')
    print('  VERITE               → github.com/stevejpapad/image-text-verification')
    print('  MMFakeBench          → huggingface.co/datasets/Xuannan/MMFakeBench')
print('=' * 50)

  39 / 39 checks passed
  ✓  All data verified — ready for EDA.
  Next: notebooks/01_setup/eda.ipynb


In [56]:
#python -c "
#from huggingface_hub import login, snapshot_download
#login(token='REMOVED_HF_TOKEN')
#snapshot_download(
 #   repo_id='liuxuannan/MMFakeBench',
 #   repo_type='dataset',
 #   local_dir='F:/GitHub/eve-rag/data/mmfakebench'
#)
#"